# Study Area and Dataset: Meta-IRL for Arctic Ship Navigation

**Companion notebook 01 — accompanying paper.**

This project learns navigation preferences of Arctic-going commercial vessels via
**meta inverse reinforcement learning (meta-IRL)**. The environment is a tabular,
goal-conditioned MDP on an **H3 resolution-6 hexagonal graph of Arctic water cells**
(Canadian Arctic / Northwest Passage region). Demonstrations are **AIS voyages of
cargo and tanker vessels** (2016–2024, July–October shipping seasons), and per-cell
environmental context comes from **ERA5** (atmosphere) and **ORAS5** (ocean / sea ice)
reanalyses aggregated onto the same hex grid.

This notebook documents the study area and the demonstration dataset:

1. **Study-area map** — the navigable water graph (14,206 H3 res-6 cells).
2. **Voyage "spaghetti" map** — all raw voyages as polylines, colored by vessel category.
3. **Seasonal sea-ice contrast** — ORAS5 ice concentration painted per hex for an
   early-season vs. late-season month, illustrating how the feasible route set changes.
4. **Dataset statistics** — voyages per category / year / month, unique vessels,
   episode lengths, and train/val/test/temporal-shift split sizes.
5. **Corrected chance baseline** — the per-decision masked-uniform log-likelihood on
   the test split, which corrects the naive $\log(1/6) \approx -1.792$ figure.

All figures are saved at 300 dpi to `notebooks/figures/fig01_*.png`.

In [ ]:
import sys
sys.path.insert(0, "..")  # package root (kernel cwd = notebooks/)

# NOTE: `arctic_meta_irl.data` must be imported BEFORE `.env` / `.features`
# (circular-import sensitivity in the package).
import arctic_meta_irl.data  # noqa: F401  (import-order requirement)

import json
import pickle
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.collections import LineCollection, PolyCollection
import h3
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from arctic_meta_irl.env.graph_mdp import GraphMDP
from arctic_meta_irl.data.dataset import load_episodes
from arctic_meta_irl.data.splits import load_splits

DATA = Path("..") / "data"
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

# --- publication style -------------------------------------------------
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# --- load cached artefacts ---------------------------------------------
mdp = GraphMDP.load(DATA / "cache" / "mdp_r6.npz")
episodes = load_episodes(DATA / "cache" / "episodes.pkl")
splits = load_splits(DATA / "cache" / "splits.json")
with open(DATA / "raw" / "timestamped_voyages_all.pkl", "rb") as f:
    voyages = pickle.load(f)

print(f"MDP: {mdp.n_states:,} states (H3 res-6 water cells), "
      f"{mdp.n_actions} actions (stay + up to 6 moves)")
print(f"Raw voyages: {len(voyages):,}   |   mapped episodes: {len(episodes):,}")
print("Splits (episodes):", {k: len(v) for k, v in splits.items()})

lat, lon = mdp.latlng[:, 0], mdp.latlng[:, 1]
print(f"Grid extent: lat [{lat.min():.1f}, {lat.max():.1f}] deg N, "
      f"lon [{lon.min():.1f}, {lon.max():.1f}] deg E")

## 1. Study area: the H3 res-6 navigation graph

States of the MDP are the centroids of 14,206 H3 resolution-6 water cells
(≈36 km² each) covering the Canadian Arctic Archipelago, Hudson Bay, Baffin Bay
and adjacent waters. Actions move a vessel to one of at most six neighboring
cells (plus *stay*); coastal cells and pentagon-adjacent cells have fewer
admissible moves, which the MDP exposes through a boolean action mask.

The map uses a North Polar Stereographic projection centered on the data. If
Natural Earth coastline data cannot be downloaded (offline execution), we fall
back to the hex cells alone — the water cells themselves trace the coastline.

In [ ]:
PROJ = ccrs.NorthPolarStereo(central_longitude=-96)
PC = ccrs.PlateCarree()
EXTENT = [lon.min() - 2.5, lon.max() + 2.5, lat.min() - 1.2, lat.max() + 1.0]


def make_map(ax, land_color="0.85", coast_lw=0.5):
    """Set extent and (best-effort) add Natural Earth land + coastlines."""
    ax.set_extent(EXTENT, crs=PC)
    try:  # Natural Earth download can fail when offline
        ax.add_feature(cfeature.LAND.with_scale("50m"),
                       facecolor=land_color, edgecolor="none", zorder=0)
        ax.add_feature(cfeature.COASTLINE.with_scale("50m"),
                       linewidth=coast_lw, edgecolor="0.45", zorder=3)
    except Exception as e:  # offline fallback: hex cells outline the coast
        warnings.warn(f"Coastline features unavailable ({e}); "
                      "plotting without Natural Earth layers.")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.7",
                      alpha=0.6, linestyle=":")
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 8, "color": "0.35"}
    return ax


fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=PROJ)
make_map(ax)
ax.scatter(lon, lat, s=0.9, c="#9fb6c9", marker="h", linewidths=0,
           transform=PC, zorder=2, rasterized=True)

# label a few orientation landmarks
landmarks = {
    "Hudson Bay": (59.5, -85.5), "Baffin Bay": (73.0, -68.5),
    "Beaufort Sea": (71.8, -133.0), "Foxe Basin": (66.7, -79.0),
    "Viscount Melville Sd.": (74.4, -108.0),
}
for name, (la, lo) in landmarks.items():
    ax.text(lo, la, name, transform=PC, fontsize=8, style="italic",
            color="0.25", ha="center", zorder=5)

ax.set_title("Study area: H3 resolution-6 navigable-water graph "
             f"({mdp.n_states:,} cells)", pad=12)
ax.text(0.01, 0.01, "Cell centroids shown; each hex cell ≈ 36 km²",
        transform=ax.transAxes, fontsize=8, color="0.4")
fig.savefig(FIGDIR / "fig01_study_area.png")
plt.show()

## 2. Demonstrations: AIS voyage "spaghetti" map

Every raw voyage is drawn as a polyline through the centroids of its H3 cell
sequence, colored by vessel category. Thin, semi-transparent lines make traffic
density visible: the **Northwest Passage corridors**, the **Hudson Bay /
Hudson Strait route**, and coastal resupply routes emerge directly from the data.

In [ ]:
cat_style = {  # plot order: dense first, sparse on top
    "cargo":  dict(color="#1f77b4", lw=0.45, alpha=0.30, z=2),
    "tanker": dict(color="#d95f02", lw=0.40, alpha=0.28, z=3),
    "other":  dict(color="#2ca02c", lw=0.55, alpha=0.45, z=4),
}
cat_counts = Counter(v["category"] for v in voyages)

# cache centroids for every cell that appears in any voyage
cent = {}
for v in voyages:
    for c in v["cells"]:
        if c not in cent:
            cent[c] = h3.cell_to_latlng(c)  # (lat, lng)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=PROJ)
make_map(ax)

for cat, st in cat_style.items():
    segs = []
    for v in voyages:
        if v["category"] != cat or len(v["cells"]) < 2:
            continue
        pts = np.array([(cent[c][1], cent[c][0]) for c in v["cells"]])  # (lon, lat)
        segs.append(pts)
    lc = LineCollection(segs, colors=st["color"], linewidths=st["lw"],
                        alpha=st["alpha"], transform=PC, zorder=st["z"])
    ax.add_collection(lc)

handles = [plt.Line2D([], [], color=st["color"], lw=2,
                      label=f"{cat} (n={cat_counts[cat]:,})")
           for cat, st in cat_style.items()]
ax.legend(handles=handles, loc="lower left", fontsize=9,
          title=f"{len(voyages):,} voyages, "
                f"{len({v['mmsi'] for v in voyages})} vessels",
          title_fontsize=9)
ax.set_title("AIS demonstration voyages, 2016–2024 (Jul–Oct shipping seasons)",
             pad=12)
fig.savefig(FIGDIR / "fig01_voyages_by_category.png")
plt.show()

## 3. Seasonal sea-ice contrast (ORAS5 ice concentration)

ORAS5 monthly sea-ice concentration is aggregated per hex cell. ORAS5 layers are
available for the July–October shipping season of each year; we contrast
**July** (late break-up, extensive residual ice) against **September**
(annual sea-ice minimum) of 2024, on a shared color scale.

In [ ]:
MONTH_A, MONTH_B = "202407", "202409"   # early vs. late season, same year
for m in (MONTH_A, MONTH_B):
    assert (DATA / "raw" / "oras5_h3" / f"oras5_h3_{m}.pkl").exists(), m


def load_oras5_conc(yyyymm):
    with open(DATA / "raw" / "oras5_h3" / f"oras5_h3_{yyyymm}.pkl", "rb") as f:
        d = pickle.load(f)
    conc = dict(zip(d["cells"], d["ice_conc"]))
    return np.array([conc.get(c, np.nan) for c in mdp.cells])


# hex boundary polygons in (lon, lat), computed once
hex_polys = [np.array([(p[1], p[0]) for p in h3.cell_to_boundary(c)])
             for c in mdp.cells]

conc = {m: load_oras5_conc(m) for m in (MONTH_A, MONTH_B)}
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
cmap = plt.get_cmap("Blues_r")

fig = plt.figure(figsize=(13, 5.8))
for k, m in enumerate((MONTH_A, MONTH_B)):
    ax = fig.add_subplot(1, 2, k + 1, projection=PROJ)
    make_map(ax)
    pc = PolyCollection(hex_polys, array=conc[m], cmap=cmap, norm=norm,
                        edgecolors="none", transform=PC, zorder=2,
                        rasterized=True)
    ax.add_collection(pc)
    label = f"{m[:4]}-{m[4:]}"
    frac = np.nanmean(conc[m] > 0.15) * 100  # >15% conc ~ "ice-infested"
    ax.set_title(f"{label}   ({frac:.0f}% of cells above 15% ice conc.)",
                 pad=10)

cb = fig.colorbar(pc, ax=fig.axes, orientation="horizontal",
                  fraction=0.045, pad=0.06, aspect=45)
cb.set_label("ORAS5 sea-ice concentration (fraction)")
fig.suptitle("Seasonal sea-ice contrast on the navigation grid", y=0.93,
             fontsize=12)
fig.savefig(FIGDIR / "fig01_ice_seasonal.png")
plt.show()

d_mean = np.nanmean(conc[MONTH_A]) - np.nanmean(conc[MONTH_B])
print(f"Takeaway: mean ice concentration drops from "
      f"{np.nanmean(conc[MONTH_A]):.3f} (Jul) to "
      f"{np.nanmean(conc[MONTH_B]):.3f} (Sep) — the central Northwest Passage "
      f"corridors that are ice-blocked in July open up by September, "
      f"so the feasible route set (and hence expert route choice) is strongly "
      f"month-dependent.")

## 4. Dataset statistics

Composition of the raw voyage set and the mapped MDP episodes. Note that
**3,254 raw voyages reduce to 3,186 episodes**: voyages whose cell sequence
would be dropped if they could not be mapped to a contiguous path of ≥2 on-graph states (off-graph cells,
unbridgeable gaps) are dropped during dataset construction.
Splits are **vessel-level** (each MMSI in exactly one of train/val/test) with a
`temporal_shift` set holding out the most recent months of training-vessel
voyages.

In [ ]:
voy_df = pd.DataFrame([{k: v[k] for k in
                        ("mmsi", "category", "year", "month", "vessel_length")}
                       for v in voyages])

cat_tbl = (voy_df.groupby("category")
           .agg(voyages=("mmsi", "size"),
                unique_vessels=("mmsi", "nunique"),
                median_length_m=("vessel_length", "median"))
           .sort_values("voyages", ascending=False))
cat_tbl.loc["TOTAL"] = [len(voy_df), voy_df["mmsi"].nunique(),
                        voy_df["vessel_length"].median()]
cat_tbl = cat_tbl.astype({"voyages": int, "unique_vessels": int})
print("Voyages per vessel category")
display(cat_tbl)

ym = (voy_df.pivot_table(index="year", columns="month", values="mmsi",
                         aggfunc="size", fill_value=0)
      .rename(columns=lambda m: f"{m:02d}"))
ym["total"] = ym.sum(axis=1)
print("\nVoyages per year x month (Jul-Oct shipping seasons only)")
display(ym)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# --- (a) year x month heatmap ------------------------------------------
ax = axes[0]
mat = ym.drop(columns="total").values
im = ax.imshow(mat, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(mat.shape[1]),
              ["Jul", "Aug", "Sep", "Oct"])
ax.set_yticks(range(mat.shape[0]), ym.index.astype(str))
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, mat[i, j], ha="center", va="center", fontsize=8,
                color="white" if mat[i, j] > mat.max() * 0.6 else "0.2")
ax.set_title("(a) Voyages per year and month")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.spines[:].set_visible(False)

# --- (b) episode length histogram --------------------------------------
ax = axes[1]
ep_len = np.array([len(e) for e in episodes])  # number of decisions
ax.hist(ep_len, bins=np.arange(0, ep_len.max() + 10, 10),
        color="#4c72b0", edgecolor="white", linewidth=0.3)
ax.axvline(np.median(ep_len), color="#c44e52", lw=1.4, ls="--",
           label=f"median = {np.median(ep_len):.0f}")
ax.set_title(f"(b) Episode length ({len(episodes):,} episodes, "
             f"{ep_len.sum():,} decisions)")
ax.set_xlabel("Decisions per episode (state transitions)")
ax.set_ylabel("Episodes")
ax.legend()

fig.tight_layout()
fig.savefig(FIGDIR / "fig01_dataset_statistics.png")
plt.show()

print(f"Episode length: mean {ep_len.mean():.1f}, median {np.median(ep_len):.0f}, "
      f"p95 {np.percentile(ep_len, 95):.0f}, max {ep_len.max()}")
print(f"Note: the small spike at 512 decisions ({int((ep_len == 512).sum())} "
      f"episodes) is the dataset builder's max_horizon truncation, not a "
      f"natural voyage-length mode.")

In [ ]:
ep_by_vidx = {e.voyage_index: e for e in episodes}

rows = []
for name, idxs in splits.items():
    eps = [ep_by_vidx[i] for i in idxs if i in ep_by_vidx]
    rows.append({
        "split": name,
        "episodes": len(eps),
        "decisions": int(sum(len(e) for e in eps)),
        "unique_vessels": len({e.mmsi for e in eps}),
        "share_episodes_%": 100 * len(eps) / len(episodes),
    })
split_tbl = pd.DataFrame(rows).set_index("split")
split_tbl["share_episodes_%"] = split_tbl["share_episodes_%"].round(1)
print("Vessel-level splits (temporal_shift = most recent months of "
      "training-vessel voyages)")
display(split_tbl)

assert split_tbl["episodes"].sum() == len(episodes), "splits must partition episodes"

## 5. Corrected chance baseline (masked-uniform log-likelihood)

A common chance baseline for a 6-neighbor hex grid is $\log(1/6) \approx -1.792$
nats per decision. That figure is **incorrect for this MDP** for two reasons:
the action set includes *stay* (7 actions where all are admissible), and coastal
cells have fewer than 6 admissible moves. The correct chance baseline is the
**masked-uniform policy**: at each visited decision state $s$, the policy places
probability $1/|\mathcal{A}(s)|$ on each admissible action, giving

$$\mathrm{LL}_{\text{chance}} \;=\; \frac{1}{N}\sum_{(s,a) \in \mathcal{D}_{\text{test}}} -\log |\mathcal{A}(s)|.$$

We compute it over every decision of the **test split**.

In [ ]:
n_admissible = mdp.action_mask.sum(axis=1)  # |A(s)| per state

lls = []
for i in splits["test"]:
    ep = ep_by_vidx.get(i)
    if ep is None:
        continue
    lls.append(-np.log(n_admissible[ep.states[:-1]]))
lls = np.concatenate(lls)

ll_masked_uniform = lls.mean()
print(f"Test decisions:                 {lls.size:,}")
print(f"Masked-uniform chance LL:       {ll_masked_uniform:.4f} nats/decision")
print(f"Naive log(1/6) baseline:        {np.log(1/6):.4f} nats/decision")
print(f"Naive log(1/7) (all 7 actions): {np.log(1/7):.4f} nats/decision")
print()
print(f"=> The paper's chance baseline should be {ll_masked_uniform:.3f}, "
      f"not {np.log(1/6):.3f}: most interior cells expose 7 admissible "
      f"actions (stay + 6 moves), pushing chance below log(1/6), while "
      f"coastal masking pulls it back up slightly.")

# distribution of admissible-action counts over visited test states
visited = np.concatenate([ep_by_vidx[i].states[:-1]
                          for i in splits["test"] if i in ep_by_vidx])
dist = pd.Series(Counter(n_admissible[visited]), name="decisions").sort_index()
dist.index.name = "|A(s)| admissible actions"
display(dist.to_frame().assign(share_pct=lambda d:
        (100 * d["decisions"] / d["decisions"].sum()).round(2)))

## Summary

- **Environment**: 14,206 H3 res-6 water cells spanning ~60–77° N, 128–64.5° W;
  deterministic 7-action MDP (stay + up to 6 canonical-bearing moves) with
  action masking at coasts.
- **Demonstrations**: 3,254 AIS voyages (2,375 cargo / 811 tanker / 68 other)
  from 209 vessels, Jul–Oct 2016–2024; 3,186 map to valid MDP episodes
  (the 68 `other`-category voyages are removed by the configured `data.categories: [cargo, tanker]` filter; kept episodes are exactly 2,375 cargo + 811 tanker, i.e. **zero** voyages were lost to graph mapping).
- **Splits**: vessel-disjoint train/val/test (1,939 / 246 / 833 episodes) plus a
  168-episode temporal-shift set.
- **Seasonality**: ORAS5 ice concentration shows the feasible route set is
  strongly month-dependent (July vs. September), motivating month-conditioned
  features and temporal-shift evaluation.
- **Chance baseline**: the masked-uniform per-decision log-likelihood on the
  test split is **≈ −1.896 nats**, correcting the naive $\log(1/6) = −1.792$
  figure used previously.